In [212]:
import sys
sys.path.append('./Textual-Anomaly-Detection-Framework/Anomaly Detection Framework')

from Data_Preparation.Embedding import embedding_encoder
from Data_Preparation.Tac import tac
from Data_Preparation import utils
from Modelisation.FlowMatching import flow_matching
from Modelisation.Baselines.OCSVM import ocsvm
from Modelisation.Baselines.CVDD.utils import build_vocab, cvdd_model_pipeline
import Modelisation.evaluation as ev
from Modelisation.Baselines.CVDD.networks import cvdd_Net
from utils import save_results


import torch
from torch import Tensor
from torch.utils.data import TensorDataset, DataLoader
import optuna
import torch
from torch import nn, Tensor
import numpy as np
from transformers import AutoTokenizer
from datasets import Dataset, concatenate_datasets

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [213]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [449]:
# train_20ng_, test_20ng_ = utils.import_dataset(name="20newsgroups", batch_size=BATCH_SIZE)
train_reuters_, test_reuters_ = utils.import_dataset(name="20newsgroups", batch_size=BATCH_SIZE)
# train_wos = utils.import_dataset(name="WOS", batch_size=BATCH_SIZE)
# train_dbpedia14, test_dbpedia14 = utils.import_dataset(name="DBpedia14", batch_size=BATCH_SIZE)
# train_agnews, test_agnews = utils.import_dataset(name="AGNews", batch_size=BATCH_SIZE)

20newsgroups dataset importing .... 




Repo card metadata block was not found. Setting CardData to empty.


In [450]:
train_reuters_ = utils.preprocess(train_reuters_.dataset)
test_reuters_ = utils.preprocess(test_reuters_.dataset)

In [451]:
def train_test_val_split(train, test, inlier_topic, dataset_name, type_tac, anomaly_rate, verbose=False):
    
    train_inlier, train_anomaly = tac.textual_anomaly_contamination(train, dataset_name, inlier_topic, type_tac, anomaly_rate, True)

    if verbose:
        print("TRAINSET")
        print(train_inlier)
        print(train_anomaly)

    n_inliers_val = int(0.1 * len(train_inlier))
    inlier_indices = np.random.choice(len(train_inlier), n_inliers_val, replace=False)
    val_inlier_dataset = train_inlier.select(inlier_indices)

    train_inlier = train_inlier.select([i for i in range(len(train_inlier)) if i not in inlier_indices])

    n_anomalies_val = int(n_inliers_val / 0.9 * 0.1)
    anomaly_indices = np.random.choice(len(train_anomaly), n_anomalies_val, replace=False)
    val_anomaly_dataset = train_anomaly.select(anomaly_indices)

    val_ = concatenate_datasets([val_inlier_dataset, val_anomaly_dataset]).shuffle(seed=42)
    
    if verbose:
        print("\nVALSET")
        print(val_)
        print()

    test_ = tac.textual_anomaly_contamination(test, dataset_name, inlier_topic, type_tac, anomaly_rate, False)
    if verbose:
        print("TESTSET")
        print(test_)

    return train_inlier, train_anomaly, val_, test_

In [452]:
inlier_topic = 'science'
dataset_name = '20newsgroups'
type_tac = 'ruff' 
anomaly_rate = 0.1

train_inlier_reuters, train_anomaly_reuters, val_reuters, test_reuters = train_test_val_split(train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate, True)

TRAINSET
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 2311
})
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 256
})

VALSET
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 256
})

TESTSET
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 1690
})


In [453]:
model_name = 'all-MiniLM-L6-v2'
sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

# model_name = 'distilbert-base-uncased'

# bertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'bert')

# model_name = 'glove_300d.kv'

# gloveEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'glove')

In [454]:
train_inlier_reuters = sentencebertEncoder.forward(train_inlier_reuters)
# train_anomaly_dl_20ng = sentencebertEncoder.forward(train_anomaly_dl_20ng)

test_reuters = sentencebertEncoder.forward(test_reuters)
# test_dl_20ng = sentencebertEncoder.forward(test_dl_20ng)

val_reuters = sentencebertEncoder.forward(val_reuters)
# val_20ng = sentencebertEncoder.forward(val_20ng

In [455]:
# X_inlier = Tensor(train_inlier_dl_20ng['sbert_embeddings']).to(device)
X_inlier = Tensor(train_inlier_reuters['sbert_embeddings']).to(device)
# X_inlier = Tensor(train_inlier_dl_20ng['glove_embedding']).to(device)

X_test =  Tensor(test_reuters['sbert_embeddings']).to(device)
y_test = np.array(test_reuters['anomaly_class'])

X_val =  Tensor(val_reuters['sbert_embeddings']).to(device)
y_val = np.array(val_reuters['anomaly_class'])

print(X_inlier.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

torch.Size([2080, 384])
torch.Size([1690, 384])
(1690,)
torch.Size([256, 384])
(256,)


## FM 

In [457]:
batch_size_default = 32
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size_default, shuffle=True)
input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def objective(trial):

    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    # n_epochs = trial.suggest_int("n_epochs", 1000, 2000, step=100)
    n_epochs = trial.suggest_int("n_epochs", 100, 500, step=50)
    # source = trial.suggest_categorical("source", ["sphere", "sphere-noised"])
    source = trial.suggest_categorical("source", ["gaussian", "sphere", "sphere-noised"])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0, 1e-3, log=False)

    dl_train = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    flow_model = flow_matching.FlowMatching(source, X_inlier.cpu(), input_dim, latent_dim, sinu, device).to(device)

    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    print("\n############################################################")
    print(f"batch_size: {batch_size} | n_epochs:{n_epochs} | source: {source} | lr: {lr} | weight_decay: {weight_decay}\n")
    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)
    flow_model_trained = fm_trainer.train(dl_train, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')
    

    auc, fpr95, ap = fm_trainer.test(X_val, y_val, score_type='norm', solver_type='midpoint', n_steps=10)

    score = auc + ap - fpr95
    
    print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f} | SCORE: {score: .4f}")  
    
#     ocsvm_kwargs = {
#         "nu": 0.1,
#         "kernel": 'rbf',
#         "gamma": 'scale'
#         }
#     clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

#     _ = clf.predict(X_test.cpu().detach())           
#     scores_val = clf.decision_function(X_val.cpu().detach())

#     auc, ap, fpr95 = ev.evaluation(y_val, scores_val, verbose=False)
#     print(f"OCSVM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")
    
    return score


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  

print("Best hyperparameters:", study.best_params)
print("Best score:", study.best_value)

[I 2025-11-26 18:10:22,283] A new study created in memory with name: no-name-b60686c2-77a3-409a-ae40-bcac23a16ab6



############################################################
batch_size: 128 | n_epochs:450 | source: sphere-noised | lr: 3.673559553260436e-05 | weight_decay: 0.0006756040958035828

 step 0 -> loss : 0.06912
 step 90 -> loss : 0.06750
 step 180 -> loss : 0.06759
 step 270 -> loss : 0.06761
 step 360 -> loss : 0.06902


[I 2025-11-26 18:10:41,846] Trial 0 finished with value: 0.44139002141449957 and parameters: {'batch_size': 128, 'n_epochs': 450, 'source': 'sphere-noised', 'lr': 3.673559553260436e-05, 'weight_decay': 0.0006756040958035828}. Best is trial 0 with value: 0.44139002141449957.


AUC: 0.7754 | FPR@95: 0.6104 | AP: 0.2764
FM --> AUC: 0.7754 | FPR@95: 0.6104 | AP: 0.2764 | SCORE:  0.4414

############################################################
batch_size: 64 | n_epochs:100 | source: sphere | lr: 0.005780930431028793 | weight_decay: 0.0009883006882292651

 step 0 -> loss : 0.00500
 step 20 -> loss : 0.00509
 step 40 -> loss : 0.00507
 step 60 -> loss : 0.00500
 step 80 -> loss : 0.00505


[I 2025-11-26 18:10:46,177] Trial 1 finished with value: 0.409782870764583 and parameters: {'batch_size': 64, 'n_epochs': 100, 'source': 'sphere', 'lr': 0.005780930431028793, 'weight_decay': 0.0009883006882292651}. Best is trial 0 with value: 0.44139002141449957.


...

AUC: 0.8864 | FPR@95: 0.5022 | AP: 0.5430
FM --> AUC: 0.8864 | FPR@95: 0.5022 | AP: 0.5430 | SCORE:  0.9272

############################################################
batch_size: 64 | n_epochs:500 | source: sphere-noised | lr: 1.7260568040906227e-05 | weight_decay: 0.0009713322172879118

 step 0 -> loss : 0.06924
 step 100 -> loss : 0.06772
 step 200 -> loss : 0.06765
 step 300 -> loss : 0.06804
 step 400 -> loss : 0.06761


[I 2025-11-26 18:13:29,362] Trial 8 finished with value: 0.43404859673815155 and parameters: {'batch_size': 64, 'n_epochs': 500, 'source': 'sphere-noised', 'lr': 1.7260568040906227e-05, 'weight_decay': 0.0009713322172879118}. Best is trial 7 with value: 0.927220366921908.


AUC: 0.7752 | FPR@95: 0.6190 | AP: 0.2779
FM --> AUC: 0.7752 | FPR@95: 0.6190 | AP: 0.2779 | SCORE:  0.4340

############################################################
batch_size: 32 | n_epochs:200 | source: sphere-noised | lr: 0.001278179575562745 | weight_decay: 0.0008447661445887043

 step 0 -> loss : 0.06913
 step 40 -> loss : 0.06691
 step 80 -> loss : 0.06767
 step 120 -> loss : 0.06767
 step 160 -> loss : 0.06871


[I 2025-11-26 18:13:45,628] Trial 9 finished with value: 0.3724080403543498 and parameters: {'batch_size': 32, 'n_epochs': 200, 'source': 'sphere-noised', 'lr': 0.001278179575562745, 'weight_decay': 0.0008447661445887043}. Best is trial 7 with value: 0.927220366921908.


AUC: 0.7754 | FPR@95: 0.6580 | AP: 0.2550
FM --> AUC: 0.7754 | FPR@95: 0.6580 | AP: 0.2550 | SCORE:  0.3724
Best hyperparameters: {'batch_size': 64, 'n_epochs': 400, 'source': 'sphere-noised', 'lr': 0.0024612273606883254, 'weight_decay': 1.2993842114474074e-05}
Best score: 0.927220366921908


In [458]:
list_auc_fm = []
list_fpr_fm = []
list_ap_fm = []
list_auc_ocsvm = []
list_fpr_ocsvm = []
list_ap_ocsvm = []

for i in range(5):
    
    print("\n##################################")
    print(f"Loading Dataset for the run {i+1}")
    
    inlier_topic = 'science'
    dataset_name = '20newsgroups'
    type_tac = 'ruff' 
    anomaly_rate = 0.1

    train_inlier_reuters, train_anomaly_reuters, val_reuters, test_reuters = train_test_val_split(train_reuters_, test_reuters_, inlier_topic, dataset_name, type_tac, anomaly_rate, False)
    
    # print("A sample for valset : ")
    # print(val_20ng[0]['text'])
    # print()
    # print("\nVALSET")
    # print(val_20ng.num_rows)
    # print()
    print("A sample for testset : ")
    print(test_reuters[-1]['text'][:50])
    print()
    # print("TESTSET")
    # print(test_20ng.num_rows)

    model_name = 'all-MiniLM-L6-v2'

    sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

    train_inlier_reuters_emb = sentencebertEncoder.forward(train_inlier_reuters)
    test_reuters_emb = sentencebertEncoder.forward(test_reuters)
    val_reuters_emb = sentencebertEncoder.forward(val_reuters)

    X_inlier = Tensor(train_inlier_reuters_emb['sbert_embeddings']).to(device)
    # print(X_inlier.shape)

    X_test =  Tensor(test_reuters_emb['sbert_embeddings']).to(device)
    y_test = np.array(test_reuters_emb['anomaly_class'])
    # print(X_test.shape, y_test.shape)
    
    X_val =  Tensor(val_reuters_emb['sbert_embeddings']).to(device)
    y_val = np.array(val_reuters_emb['anomaly_class'])
    # print(X_val.shape, y_val.shape)
    
    
    #################################################
    ################# FLOW MATCHING #################
    #################################################
    
    # batch_size = 32
    batch_size = study.best_params['batch_size']
    X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    input_dim = X_inlier.shape[1]
    latent_dim = 256
    sinu = False
    # lr = 1e-5
    lr = study.best_params['lr']
    # weight_decay = 1e-4
    weight_decay = study.best_params['weight_decay']
    # n_epochs = 2000
    n_epochs = study.best_params['n_epochs']


    target = X_inlier.cpu()
    # source = 'sphere-noised'
    source = study.best_params['source']


    flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

    flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

    auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)
    
    list_auc_fm.append(auc)
    list_fpr_fm.append(fpr95)    
    list_ap_fm.append(ap)    
     
    #########################################
    ################# OCSVM #################
    #########################################  
    
    ocsvm_kwargs = {
        "nu": 0.1,
        "kernel": 'rbf',
        "gamma": 'scale'
        }
    clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

    _ = clf.predict(X_test.cpu().detach())           
    scores_test = clf.decision_function(X_test.cpu().detach())

    auc, ap, fpr95 = ev.evaluation(y_test, scores_test, verbose=False)
    print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")
    
    list_auc_ocsvm.append(auc)
    list_fpr_ocsvm.append(fpr95)    
    list_ap_ocsvm.append(ap)  


##################################
Loading Dataset for the run 1
A sample for testset : 
ive never ridden pillion very much but recently ha

 step 0 -> loss : 0.06015
 step 80 -> loss : 0.03729
 step 160 -> loss : 0.03703
 step 240 -> loss : 0.04086
 step 320 -> loss : 0.03759
AUC: 0.8073 | FPR@95: 0.6818 | AP: 0.3345
AUC: 0.7451 | FPR@95: 0.7765 | AP: 0.2894

##################################
Loading Dataset for the run 2
A sample for testset : 
detailed explanation deleted indeed you have struc

 step 0 -> loss : 0.05865
 step 80 -> loss : 0.04133
 step 160 -> loss : 0.04206
 step 240 -> loss : 0.03998
 step 320 -> loss : 0.04000
AUC: 0.7988 | FPR@95: 0.6450 | AP: 0.3199
AUC: 0.7676 | FPR@95: 0.6949 | AP: 0.2792

##################################
Loading Dataset for the run 3
A sample for testset : 
this is the ap story from fri morning as the walls

 step 0 -> loss : 0.05879
 step 80 -> loss : 0.04019
 step 160 -> loss : 0.03838
 step 240 -> loss : 0.03799
 step 320 -> loss : 0.0

In [459]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="flow-matching",
    auc_mean=np.mean(list_auc_fm),
    ap_mean=np.mean(list_ap_fm),
    fpr_mean=np.mean(list_fpr_fm),
    auc_std = np.std(list_auc_fm),
    ap_std =  np.std(list_ap_fm),
    fpr_std = np.std(list_fpr_fm) 
)

Résultats existants non modifiés pour (20newsgroups, science, sentence_bert, flow-matching).


In [422]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="ocsvm",
    auc_mean=np.mean(list_auc_ocsvm),
    ap_mean=np.mean(list_ap_ocsvm),
    fpr_mean=np.mean(list_fpr_ocsvm),
    auc_std = np.std(list_auc_ocsvm),
    ap_std =  np.std(list_ap_ocsvm),
    fpr_std = np.std(list_fpr_ocsvm) 
)

Nouveaux résultats ajoutés pour (reuters, trade, sentence_bert, ocsvm).


In [ ]:
{'batch_size': 128, 'n_epochs': 1500, 'source': 'sphere-noised', 'lr': 0.0018322231805160782, 'weight_decay': 1.6294578771550495e-05}

In [447]:

batch_size = 32
# batch_size = study.best_params['batch_size']
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
lr = 1e-5
# lr = study.best_params['lr']
weight_decay = 1e-4
# weight_decay = study.best_params['weight_decay']
n_epochs = 1000
# n_epochs = study.best_params['n_epochs']


target = X_inlier.cpu()
source = 'sphere-noised'
# source = study.best_params['source']


flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.MSELoss()

fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)

 step 0 -> loss : 0.06557
 step 200 -> loss : 0.06982
 step 400 -> loss : 0.06521
 step 600 -> loss : 0.06435
 step 800 -> loss : 0.05944
AUC: 0.9151 | FPR@95: 0.3388 | AP: 0.7126


## Basalines

### OCSVM

In [448]:
ocsvm_kwargs = {
        "nu": 0.1,
        "kernel": 'rbf',
        "gamma": 'scale'
        }
clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

_ = clf.predict(X_test.cpu().detach())           
scores_test = clf.decision_function(X_test.cpu().detach())

auc, ap, fpr95 = ev.evaluation(y_test, scores_test, verbose=False)
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

AUC: 0.9374 | FPR@95: 0.4876 | AP: 0.8225


### CVDD

In [14]:
type_emb = 'glove'
emb_model = 'distilbert-base-uncased'
attention_size = 150
n_attention_heads = 10
lr = 0.01
lr_milestones = (5, 8)
n_epochs = 10
lambda_p = 1.0
alpha_scheduler = 'logarithmic'

In [15]:
if type_emb == 'bert':
    tokenizer = AutoTokenizer.from_pretrained(emb_model)
    vocab = None

elif type_emb in ('glove', 'fasttext'):
    corpus = train_inlier_dl_20ng['text']
    vocab = build_vocab(corpus,min_freq=1)
    tokenizer = None

In [16]:
cvdd_model, dl_train, dl_test = cvdd_model_pipeline(train_inlier_dl_20ng, test_dl_20ng, attention_size, n_attention_heads, 
                                               type_emb, 500, 64, True, device, tokenizer, vocab)

In [17]:
cvdd_trainer = cvdd_Net.CVDDTrainer(optimizer_name='adam', learning_rate=lr, lr_milestones=lr_milestones,
                                    n_epochs=n_epochs, lambda_p=lambda_p,
                                    alpha_scheduler=alpha_scheduler, weight_decay=1e-4, device=device)

model_trained = cvdd_trainer.train(cvdd_model, dl_train)

Starting training...
KMean starts
KMeans finish
| Epoch: 001/010 | Train Time: 0.517s | Train Loss: 0.139542 |
| Epoch: 002/010 | Train Time: 0.444s | Train Loss: 0.056700 |
| Epoch: 003/010 | Train Time: 0.441s | Train Loss: 0.049708 |
| Epoch: 004/010 | Train Time: 0.439s | Train Loss: 0.046049 |
| Epoch: 005/010 | Train Time: 0.441s | Train Loss: 0.042732 |
| Epoch: 006/010 | Train Time: 0.439s | Train Loss: 0.041651 |
| Epoch: 007/010 | Train Time: 0.439s | Train Loss: 0.040930 |
| Epoch: 008/010 | Train Time: 0.439s | Train Loss: 0.040041 |
| Epoch: 009/010 | Train Time: 0.439s | Train Loss: 0.039830 |
| Epoch: 010/010 | Train Time: 0.441s | Train Loss: 0.039731 |
Training Time: 5.272s
Finished training. 



In [18]:
auc, ap, fpr95, _ = cvdd_trainer.test(model_trained, dl_test, ad_score='context_dist_mean')
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

AUC: 0.5008 | FPR@95: 0.9654 | AP: 0.1162
